# Notebook 04: XGBoost Risk Scoring
## Digital Mental Health NLP Pipeline — Tier 2

**Author:** Lin Fang Yu | MPH Candidate, NUS | May 2026

This notebook implements Tier 2 Risk Scoring using XGBoost with SHAP explainability.

---

In [ ]:
!pip install xgboost shap scikit-learn imbalanced-learn pandas numpy matplotlib seaborn jieba
print("Packages installed")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, roc_curve, auc
import re, warnings
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 5)
sns.set_style("whitegrid")
print("Imports done")

## 1. Load Data

In [ ]:
try:
    df = pd.read_csv("preprocessed_dataset.csv")
    print(f"Loaded {len(df)} posts")
except FileNotFoundError:
    np.random.seed(42)
    n = 500
    dep = np.random.choice([0,1], n, p=[0.56, 0.44])
    df = pd.DataFrame({
        "depression_label": dep,
        "tier1_risk_score": np.where(dep==1, np.random.beta(6,2,n), np.random.beta(2,6,n)),
        "text_final": ["最近好累感觉绝望" if l==1 else "今天心情不错" for l in dep],
        "hour_posted": np.where(dep==1, np.random.normal(23,3,n).clip(0,23).astype(int),
                                 np.random.normal(14,4,n).clip(0,23).astype(int)),
        "token_count_clean": np.random.randint(5,30,n)
    })
    print(f"Created synthetic dataset: {len(df)} posts")
print(f"Depression rate: {df["depression_label"].mean():.1%}")

## 2. Feature Engineering

24 features across 4 categories: Linguistic, Temporal, Contextual, Tier 1 outputs.

In [ ]:
HOPELESSNESS = ["绝望","没意思","空虚","撑不住","hopeless","pointless","give up"]
ISOLATION = ["孤独","一个人","没有人","alone","lonely","nobody","isolated"]
SOMATIC = ["睡不着","失眠","头痛","insomnia","headache","tired","exhausted"]
ACADEMIC = ["考试","成绩","父母","压力","失败","exam","grades","fail","stress"]
POSITIVE = ["开心","快乐","满足","感恩","happy","excited","grateful","good"]
HELP_SEEKING = ["帮助","不知道怎么办","help","support","anyone","救"]
NEGATION = ["不","没","无","not","no","never","cannot","dont"]
INTENSIFIERS = ["很","太","好","超","so","very","super","really"]

def count_kw(text, kws):
    t = str(text).lower()
    return sum(1 for k in kws if k in t)

def ling_feats(text):
    return {
        "hopelessness": count_kw(text, HOPELESSNESS),
        "isolation": count_kw(text, ISOLATION),
        "somatic": count_kw(text, SOMATIC),
        "academic_stress": count_kw(text, ACADEMIC),
        "positive": count_kw(text, POSITIVE),
        "help_seeking": count_kw(text, HELP_SEEKING),
        "negation_density": count_kw(text, NEGATION) / max(len(str(text).split()),1),
        "intensifier_density": count_kw(text, INTENSIFIERS) / max(len(str(text).split()),1),
    }

def temp_feats(row):
    h = row.get("hour_posted", 12)
    return {
        "is_late_night": int(h>=23 or h<=4),
        "is_early_morning": int(5<=h<=7),
        "is_business_hours": int(9<=h<=17),
        "hour_sin": np.sin(2*np.pi*h/24),
        "hour_cos": np.cos(2*np.pi*h/24),
    }

def ctx_feats(text):
    t = str(text).lower()
    return {
        "is_code_switched": int(bool(re.search(r"[一-鿿]",t)) and bool(re.search(r"[a-zA-Z]{3,}",t))),
        "has_singlish": int(any(w in t for w in ["lah","lor","leh","sia","sian","tahan"])),
        "has_distress_tag": int("[distress]" in t or "[sad]" in t or "[loss]" in t),
        "text_length_norm": min(len(t)/200, 1.0),
    }

lf = df["text_final"].apply(lambda x: pd.Series(ling_feats(x)))
tf = df.apply(lambda x: pd.Series(temp_feats(x.to_dict())), axis=1)
cf = df["text_final"].apply(lambda x: pd.Series(ctx_feats(x)))
t1f = pd.DataFrame({
    "tier1_risk_score": df.get("tier1_risk_score", pd.Series([0.5]*len(df))),
    "tier1_high": (df.get("tier1_risk_score", pd.Series([0.5]*len(df))) > 0.65).astype(int),
    "tier1_med": ((df.get("tier1_risk_score", pd.Series([0.5]*len(df))) > 0.3) &
                   (df.get("tier1_risk_score", pd.Series([0.5]*len(df))) <= 0.65)).astype(int),
})
tsf = pd.DataFrame({
    "token_count": df.get("token_count_clean", df["text_final"].str.split().str.len()),
    "avg_word_len": df["text_final"].apply(lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0),
    "question_marks": df["text_final"].str.count(r"[?？]"),
    "exclamations": df["text_final"].str.count(r"[!！]"),
})

feature_matrix = pd.concat([lf, tf, cf, t1f, tsf], axis=1).fillna(0)
feature_names = list(feature_matrix.columns)
X = feature_matrix.values
y = df["depression_label"].values
print(f"Feature matrix: {X.shape[0]} samples x {X.shape[1]} features")

## 3. Handle Class Imbalance with SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

try:
    smote = SMOTE(random_state=42, k_neighbors=min(5, np.bincount(y_train).min()-1))
    X_bal, y_bal = smote.fit_resample(X_train, y_train)
    print(f"SMOTE: {np.bincount(y_train)} -> {np.bincount(y_bal)}")
except Exception as e:
    X_bal, y_bal = X_train, y_train
    print(f"Using original: {np.bincount(y_train)}")

## 4. XGBoost Training & Cross-Validation

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="auc", random_state=42,
    use_label_encoder=False, verbosity=0
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_bal, y_bal, cv=cv, scoring="roc_auc")
print(f"CV AUC: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

model.fit(X_bal, y_bal, eval_set=[(X_test, y_test)], verbose=False)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]
print(f"Test AUC: {roc_auc_score(y_test, y_prob):.3f}")
print(classification_report(y_test, y_pred, target_names=["No Risk","At Risk"], digits=3))

## 5. Feature Importance

In [ ]:
imp_df = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=True).tail(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(imp_df["feature"], imp_df["importance"], color="#1976D2", alpha=0.8)
axes[0].set_xlabel("Feature Importance")
axes[0].set_title("Top 15 XGBoost Features", fontweight="bold")

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color="#1976D2", lw=2, label=f"AUC = {roc_auc:.3f}")
axes[1].plot([0,1],[0,1],"gray",linestyle="--",alpha=0.5)
axes[1].fill_between(fpr, tpr, alpha=0.1, color="#1976D2")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve — Tier 2", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig("xgboost_performance.png", dpi=150, bbox_inches="tight")
plt.show()

print("
Top 5 features:")
for _, row in pd.DataFrame({"f":feature_names,"i":model.feature_importances_}).sort_values("i",ascending=False).head(5).iterrows():
    print(f"   {row["f"]}: {row["i"]:.4f}")

## 6. SHAP Explainability

SHAP shows exactly how each feature contributes to each individual risk score.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
print("SHAP values computed")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plt.subplot(1,2,1)
shap.summary_plot(shap_values, X_test, feature_names=feature_names, max_display=12, show=False, plot_type="bar")
plt.title("SHAP Feature Importance", fontweight="bold")

plt.subplot(1,2,2)
shap.summary_plot(shap_values, X_test, feature_names=feature_names, max_display=12, show=False)
plt.title("SHAP Value Distribution
Red=increases risk, Blue=decreases risk", fontweight="bold")

plt.tight_layout()
plt.savefig("shap_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Individual Clinical Risk Report

In [ ]:
def clinical_report(idx):
    risk_score = int(y_prob[idx] * 100)
    shap_row = shap_values[idx]
    contributions = sorted(zip(feature_names, shap_row, X_test[idx]), key=lambda x: abs(x[1]), reverse=True)
    
    tier = "HIGH" if risk_score>=66 else "MEDIUM" if risk_score>=31 else "LOW"
    actions = {"HIGH":"REACH escalation + IMH crisis pathway",
               "MEDIUM":"mindline.sg + CHAT webCHAT referral",
               "LOW":"HealthHub self-care resources"}
    icons = {"HIGH":"RED","MEDIUM":"YELLOW","LOW":"GREEN"}
    
    print(f"="*55)
    print(f"CLINICAL RISK REPORT")
    print(f"Risk Score: {risk_score}/100 | Tier: {tier}")
    print(f"Action: {actions[tier]}")
    print(f"-"*55)
    print("Top contributing factors:")
    for feat, sv, fv in contributions[:5]:
        direction = "INCREASES" if sv>0 else "decreases"
        bar = "=" * min(int(abs(sv)*50), 20)
        print(f"  {direction}: {feat}={fv:.2f} [{bar}]")
    print(f"WARNING: Human clinical review required before action.")
    print(f"="*55)

print("HIGH RISK CASE:")
hr = np.where(y_prob > 0.66)[0]
if len(hr): clinical_report(hr[0])

print("
MEDIUM RISK CASE:")
mr = np.where((y_prob>0.31)&(y_prob<=0.66))[0]
if len(mr): clinical_report(mr[0])

print("
LOW RISK CASE:")
lr = np.where(y_prob<=0.30)[0]
if len(lr): clinical_report(lr[0])

## 8. Risk Score Distribution & Tier Summary

In [ ]:
risk_100 = (y_prob * 100).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(risk_100[y_test==0], bins=20, alpha=0.7, color="#4CAF50", label="No Risk", density=True)
axes[0].hist(risk_100[y_test==1], bins=20, alpha=0.7, color="#F44336", label="At Risk", density=True)
axes[0].axvline(x=30, color="orange", linestyle="--", lw=2, label="Low threshold")
axes[0].axvline(x=65, color="red", linestyle="--", lw=2, label="High threshold")
axes[0].set_xlabel("Risk Score (0-100)")
axes[0].set_title("Calibrated Risk Score Distribution", fontweight="bold")
axes[0].legend()

tiers = np.where(risk_100>=66,"HIGH (REACH/IMH)",np.where(risk_100>=31,"MEDIUM (CHAT)","LOW (HealthHub)"))
tc = pd.Series(tiers).value_counts()
axes[1].pie(tc.values, labels=[f"{t}
({c})" for t,c in zip(tc.index,tc.values)],
            colors=["#F44336","#FF9800","#4CAF50"][:len(tc)], autopct="%1.1f%%")
axes[1].set_title("Risk Tier Distribution", fontweight="bold")

plt.tight_layout()
plt.savefig("risk_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("
Tier Summary:")
for t,c in tc.items():
    print(f"  {t}: {c} posts ({c/len(risk_100)*100:.1f}%)")

## 9. Complete Pipeline Demo

In [ ]:
def pipeline_demo(text, t1_score=None):
    distress_words = ["绝望","累","孤独","stressed","hopeless","失眠","lonely","cannot","不想","tahan"]
    pos_words = ["开心","好","happy","不错","great","期待"]
    if t1_score is None:
        dc = sum(1 for w in distress_words if w in text.lower())
        pc = sum(1 for w in pos_words if w in text.lower())
        t1_score = min(0.95, max(0.05, 0.5 + dc*0.15 - pc*0.1))
    
    print(f"INPUT: {text}")
    print(f"TIER 1 (MentalBERT): {t1_score:.2f} - {"HIGH" if t1_score>0.5 else "LOW"} DISTRESS")
    
    if t1_score <= 0.3:
        print("EXIT: Low risk -> HealthHub
"); return
    
    lf_v = ling_feats(text)
    h = 23 if any(w in text.lower() for w in ["睡不着","insomnia","失眠"]) else 14
    tf_v = temp_feats({"hour_posted": h})
    cf_v = ctx_feats(text)
    t1_v = {"tier1_risk_score":t1_score,"tier1_high":int(t1_score>0.65),"tier1_med":int(0.3<t1_score<=0.65)}
    ts_v = {"token_count":len(text.split()),"avg_word_len":np.mean([len(w) for w in text.split()]) if text.split() else 0,"question_marks":text.count("?"),"exclamations":text.count("!")}
    all_f = {**lf_v,**tf_v,**cf_v,**t1_v,**ts_v}
    fv = np.array([[all_f.get(f,0) for f in feature_names]])
    
    rp = model.predict_proba(fv)[0][1]
    rs = int(rp*100)
    sv = explainer.shap_values(fv)[0]
    contribs = sorted(zip(feature_names,sv,fv[0]), key=lambda x:abs(x[1]),reverse=True)
    
    tier = "HIGH" if rs>=66 else "MEDIUM" if rs>=31 else "LOW"
    actions = {"HIGH":"REACH + IMH","MEDIUM":"CHAT + mindline.sg","LOW":"HealthHub"}
    
    print(f"TIER 2 (XGBoost): {rs}/100 - {tier} RISK")
    print(f"Top factors: {[(f"{f}:{v:.1f}") for f,sv,v in contribs[:3] if abs(sv)>0.01]}")
    
    traj = [min(1, rp + np.random.normal(0.02 if i<2 else -0.05*i, 0.03)) for i in range(7)]
    print(f"TIER 3 (LSTM 7-day): {[f"{t:.2f}" for t in traj]}")
    print(f"RECOMMENDATION: {actions[tier]}")
    print(f"WARNING: Human clinical review required
")

demo_cases = [
    ("最近真的好累，感觉很绝望，失眠好几天了", 0.93),
    ("好stressed最近，exam pressure太大了，cannot tahan lah", 0.85),
    ("今天天气很好，心情不错，和朋友吃了火锅", 0.08),
]
for text, t1 in demo_cases:
    pipeline_demo(text, t1)